# Coop Kitchen QMIX-lite Colab
チームの mixed return を使って戦略価値を更新する軽量ループです。

In [ ]:
from pathlib import Path
import math, json
from marl_course.algos.coop_bandit import CoopStrategyBanditPolicy
from marl_course.algos.logging import MetricLogger
from marl_course.envs.coop_kitchen import CoopKitchenEnv, builtin_layouts
from marl_course.evaluation.coop import make_team_obs


In [ ]:
config = json.loads(Path('configs/train_coop_qmix.json').read_text())
config.update({'episodes': 40, 'out': 'outputs/colab_coop_qmix', 'wandb': False})
out = Path(config['out'])


In [ ]:
policy = CoopStrategyBanditPolicy(seed=config['seed'])
layouts = list(builtin_layouts().values())
logger = MetricLogger(out, use_wandb=config['wandb'], project=config['wandb_project'], run_name='colab-coop-qmix-lite', config=config)
best_score = 0
for ep in range(config['episodes']):
    epsilon = config['epsilon_end'] + (config['epsilon_start'] - config['epsilon_end']) * math.exp(-4.0 * ep / config['episodes'])
    layout = layouts[(ep + config['seed']) % len(layouts)]
    env = CoopKitchenEnv(layout=layout)
    obs, _ = env.reset(seed=100 + ep)
    policy.reset_episode({'layout': layout.name}, epsilon=epsilon)
    done = False
    while not done:
        team_obs = make_team_obs(obs)
        actions = policy.act(team_obs, team_obs['action_mask'], deterministic=False)
        result = env.step({f'agent_{i}': actions[i] for i in range(4)})
        obs = result.observations
        done = any(result.truncations.values())
    mixed_return = env.score - 0.02 * env.collisions
    td_error = policy.update(mixed_return, alpha=config['alpha'])
    best_score = max(best_score, env.score)
    logger.log({'episode': ep, 'score': env.score, 'mixed_return': mixed_return, 'soups': env.delivered_soups, 'best_score': best_score, 'epsilon': epsilon, 'strategy': policy.current_strategy, 'td_error': td_error, 'collisions': env.collisions}, step=ep)
logger.close()


In [ ]:
out.mkdir(parents=True, exist_ok=True)
policy.save(out / 'policy.pt')
(out / 'metadata.json').write_text(json.dumps({'student_id': 'colab_coop_qmix', 'env_id': 'coop_kitchen_v1', 'algo': 'qmix-lite'}, indent=2), encoding='utf-8')
(out / 'policy.py').write_text("from pathlib import Path\nfrom marl_course.algos.coop_bandit import CoopStrategyBanditPolicy\n\ndef load_policy(model_path, device='cpu'):\n    return CoopStrategyBanditPolicy.load(Path(model_path))\n", encoding='utf-8')


In [ ]:
import pandas as pd
pd.read_csv(out / 'metrics.csv').plot(x='episode', y=['score', 'mixed_return', 'epsilon'], subplots=True, figsize=(8, 6));
